In [49]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.sparse import load_npz
from scipy.sparse import vstack
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_tfidf = load_npz("../data/processed/X_train_tfidf.npz")
X_test_tfidf = load_npz("../data/processed/X_test_tfidf.npz")

y_train_encoded = np.load("../data/processed/y_train_encoded.npy")
y_test_encoded = np.load("../data/processed/y_test_encoded.npy")
class_names = np.load("../data/processed/class_names.npy").tolist()

input_size = X_train_tfidf.shape[1]
num_classes = len(class_names)

print("Device:", device)
print("Training data:", X_train_tfidf.shape)
print("Testing data:", X_test_tfidf.shape)
print("Classes:", class_names)

Device: cpu
Training data: (24852, 50000)
Testing data: (6215, 50000)
Classes: ['cmdi', 'norm', 'path-traversal', 'sqli', 'xss']


In [50]:
class MLP(nn.Module):

    def __init__(self, input_size, num_classes):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.network(x)


model = MLP(input_size, num_classes).to(device)

print(model)

MLP(
  (network): Sequential(
    (0): Linear(in_features=50000, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=128, out_features=5, bias=True)
  )
)


In [51]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 10
batch_size = 128

mlp_history = []

for epoch in range(epochs):

    model.train()

    indices = np.random.permutation(len(y_train_encoded))
    total_loss = 0

    for start in range(0, len(indices), batch_size):

        batch_indices = indices[start:start + batch_size]

        X_batch = X_train_tfidf[batch_indices].toarray().astype(np.float32)
        X_batch = torch.from_numpy(X_batch).to(device)

        y_batch = torch.tensor(
            y_train_encoded[batch_indices],
            dtype=torch.long
        ).to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(batch_indices)

    average_loss = total_loss / len(y_train_encoded)
    mlp_history.append(average_loss)

    print(
        f"Epoch [{epoch + 1}/{epochs}], "
        f"Loss: {average_loss:.4f}"
    )

Epoch [1/10], Loss: 0.2743
Epoch [2/10], Loss: 0.0106
Epoch [3/10], Loss: 0.0051
Epoch [4/10], Loss: 0.0026
Epoch [5/10], Loss: 0.0019
Epoch [6/10], Loss: 0.0016
Epoch [7/10], Loss: 0.0015
Epoch [8/10], Loss: 0.0013
Epoch [9/10], Loss: 0.0014
Epoch [10/10], Loss: 0.0012


In [52]:
model.eval()

mlp_predictions = []

with torch.no_grad():

    for start in range(0, len(y_test_encoded), batch_size):

        X_batch = X_test_tfidf[start:start + batch_size].toarray().astype(np.float32)
        X_batch = torch.from_numpy(X_batch).to(device)

        outputs = model(X_batch)

        predictions = torch.argmax(outputs, dim=1)

        mlp_predictions.extend(predictions.cpu().numpy())

mlp_predictions = np.array(mlp_predictions)

print("Prediction count:", len(mlp_predictions))
print("Actual count:", len(y_test_encoded))

Prediction count: 6215
Actual count: 6215


In [53]:
def calculate_metrics(y_true, y_pred):

    cm = np.zeros((num_classes, num_classes), dtype=int)

    for actual, predicted in zip(y_true, y_pred):
        cm[actual, predicted] += 1

    precision_scores = []
    recall_scores = []
    f1_scores = []
    specificity_scores = []

    for i in range(num_classes):

        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp

        precision = tp / (tp + fp) if tp + fp > 0 else 0
        recall = tp / (tp + fn) if tp + fn > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0
        specificity = tn / (tn + fp) if tn + fp > 0 else 0

        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)
        specificity_scores.append(specificity)

    return {
        "accuracy": np.trace(cm) / cm.sum(),
        "balanced_accuracy": np.mean(recall_scores),
        "macro_f1": np.mean(f1_scores),
        "macro_precision": np.mean(precision_scores),
        "macro_recall": np.mean(recall_scores),
        "macro_specificity": np.mean(specificity_scores),
        "confusion_matrix": cm,
        "class_precision": precision_scores,
        "class_recall": recall_scores,
        "class_f1": f1_scores,
        "class_specificity": specificity_scores
    }

In [54]:
mlp_results = calculate_metrics(
    y_test_encoded,
    mlp_predictions
)

print("Accuracy:", round(mlp_results["accuracy"], 4))
print("Balanced Accuracy:", round(mlp_results["balanced_accuracy"], 4))
print("Macro F1:", round(mlp_results["macro_f1"], 4))
print("Macro Specificity:", round(mlp_results["macro_specificity"], 4))

print("\nClass-Based Results:")

for i, class_name in enumerate(class_names):

    print(f"\n{class_name}")
    print("Precision:", round(mlp_results["class_precision"][i], 4))
    print("Recall:", round(mlp_results["class_recall"][i], 4))
    print("F1:", round(mlp_results["class_f1"][i], 4))
    print("Specificity:", round(mlp_results["class_specificity"][i], 4))

print("\nConfusion Matrix:")
print(mlp_results["confusion_matrix"])

Accuracy: 0.9976
Balanced Accuracy: 0.966
Macro F1: 0.9732
Macro Specificity: 0.9993

Class-Based Results:

cmdi
Precision: 1.0
Recall: 0.8333
F1: 0.9091
Specificity: 1.0

norm
Precision: 0.999
Recall: 0.9972
F1: 0.9981
Specificity: 0.9983

path-traversal
Precision: 1.0
Recall: 1.0
F1: 1.0
Specificity: 1.0

sqli
Precision: 0.9991
Recall: 0.9995
F1: 0.9993
Specificity: 0.9995

xss
Precision: 0.9224
Recall: 1.0
F1: 0.9596
Specificity: 0.9985

Confusion Matrix:
[[  15    3    0    0    0]
 [   0 3850    0    2    9]
 [   0    0   58    0    0]
 [   0    1    0 2170    0]
 [   0    0    0    0  107]]


In [55]:
import os
import pandas as pd

os.makedirs("../reports", exist_ok=True)

summary_df = pd.DataFrame([{
    "Model": "MLP",
    "Loss Function": "CrossEntropy",
    "Accuracy": mlp_results["accuracy"],
    "Balanced Accuracy": mlp_results["balanced_accuracy"],
    "Macro F1": mlp_results["macro_f1"],
    "Macro Specificity": mlp_results["macro_specificity"]
}])

summary_df.to_csv(
    "../reports/mlp_baseline_results.csv",
    index=False,
    float_format="%.4f"
)

class_df = pd.DataFrame({
    "Class": class_names,
    "Precision": mlp_results["class_precision"],
    "Recall": mlp_results["class_recall"],
    "F1 Score": mlp_results["class_f1"],
    "Specificity": mlp_results["class_specificity"]
})

class_df.to_csv(
    "../reports/mlp_baseline_class_metrics.csv",
    index=False,
    float_format="%.4f"
)

confusion_df = pd.DataFrame(
    mlp_results["confusion_matrix"],
    index=class_names,
    columns=class_names
)

confusion_df.index.name = "Actual / Predicted"

confusion_df.to_csv(
    "../reports/mlp_baseline_confusion_matrix.csv"
)


display(summary_df)
display(class_df)
display(confusion_df)

,Model,Loss Function,Accuracy,Balanced Accuracy,Macro F1,Macro Specificity
0,MLP,CrossEntropy,0.997586,0.966005,0.973219,0.999267


,Class,Precision,Recall,F1 Score,Specificity
0,cmdi,1.000000,0.833333,0.909091,1.000000
1,norm,0.998962,0.997151,0.998056,0.998301
2,path-traversal,1.000000,1.000000,1.000000,1.000000
3,sqli,0.999079,0.999539,0.999309,0.999505
4,xss,0.922414,1.000000,0.959641,0.998527


,cmdi,norm,path-traversal,sqli,xss
Actual / Predicted,,,,,
cmdi,15,3,0,0,0
norm,0,3850,0,2,9
path-traversal,0,0,58,0,0
sqli,0,1,0,2170,0
xss,0,0,0,0,107


##MLP with Focal Loss

In [56]:
class FocalLoss(nn.Module):

    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()

        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):

        log_probs = torch.log_softmax(inputs, dim=1)

        log_pt = log_probs.gather(
            1, targets.unsqueeze(1)
        ).squeeze(1)

        pt = torch.exp(log_pt)

        focal_weight = (1 - pt) ** self.gamma

        if self.alpha is not None:
            alpha_weight = self.alpha[targets]
            loss = -alpha_weight * focal_weight * log_pt
        else:
            loss = -focal_weight * log_pt

        return loss.mean()

In [57]:
class_counts = np.bincount(
    y_train_encoded,
    minlength=num_classes
)

class_weights = len(y_train_encoded) / (
    num_classes * class_counts
)

alpha_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print("Class Counts:", class_counts)
print("Class Weights:", class_weights)
print("Alpha Weights:", alpha_weights)

Class Counts: [   71 15443   232  8681   425]
Class Weights: [70.0056338   0.32185456 21.42413793  0.57256076 11.69505882]
Alpha Weights: tensor([70.0056,  0.3219, 21.4241,  0.5726, 11.6951])


In [58]:
def train_focal_mlp(gamma, alpha_weights, epochs=10, batch_size=128):

    model = MLP(input_size, num_classes).to(device)

    criterion = FocalLoss(
        alpha=alpha_weights,
        gamma=gamma
    )

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    history = []

    for epoch in range(epochs):

        model.train()

        indices = np.random.permutation(len(y_train_encoded))
        total_loss = 0

        for start in range(0, len(indices), batch_size):

            batch_indices = indices[start:start + batch_size]

            X_batch = X_train_tfidf[batch_indices].toarray().astype(np.float32)
            X_batch = torch.from_numpy(X_batch).to(device)

            y_batch = torch.tensor(
                y_train_encoded[batch_indices],
                dtype=torch.long
            ).to(device)

            optimizer.zero_grad()

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(batch_indices)

        average_loss = total_loss / len(y_train_encoded)
        history.append(average_loss)

        print(
            f"Gamma: {gamma}, "
            f"Epoch [{epoch + 1}/{epochs}], "
            f"Loss: {average_loss:.4f}"
        )

    return model, history

In [59]:
gamma_values = [1.0, 2.0, 3.0]

focal_mlp_models = {}
focal_mlp_histories = {}

for gamma in gamma_values:

    print(f"\nTraining MLP with Focal Loss (Gamma = {gamma})")

    trained_model, history = train_focal_mlp(
        gamma=gamma,
        alpha_weights=alpha_weights,
        epochs=10,
        batch_size=128
    )

    focal_mlp_models[gamma] = trained_model
    focal_mlp_histories[gamma] = history




Training MLP with Focal Loss (Gamma = 1.0)
Gamma: 1.0, Epoch [1/10], Loss: 0.4026
Gamma: 1.0, Epoch [2/10], Loss: 0.0242
Gamma: 1.0, Epoch [3/10], Loss: 0.0133
Gamma: 1.0, Epoch [4/10], Loss: 0.0066
Gamma: 1.0, Epoch [5/10], Loss: 0.0053
Gamma: 1.0, Epoch [6/10], Loss: 0.0034
Gamma: 1.0, Epoch [7/10], Loss: 0.0070
Gamma: 1.0, Epoch [8/10], Loss: 0.0061
Gamma: 1.0, Epoch [9/10], Loss: 0.0077
Gamma: 1.0, Epoch [10/10], Loss: 0.0102

Training MLP with Focal Loss (Gamma = 2.0)
Gamma: 2.0, Epoch [1/10], Loss: 0.2817
Gamma: 2.0, Epoch [2/10], Loss: 0.0130
Gamma: 2.0, Epoch [3/10], Loss: 0.0053
Gamma: 2.0, Epoch [4/10], Loss: 0.0029
Gamma: 2.0, Epoch [5/10], Loss: 0.0038
Gamma: 2.0, Epoch [6/10], Loss: 0.0030
Gamma: 2.0, Epoch [7/10], Loss: 0.0065
Gamma: 2.0, Epoch [8/10], Loss: 0.0028
Gamma: 2.0, Epoch [9/10], Loss: 0.0021
Gamma: 2.0, Epoch [10/10], Loss: 0.0019

Training MLP with Focal Loss (Gamma = 3.0)
Gamma: 3.0, Epoch [1/10], Loss: 0.2237
Gamma: 3.0, Epoch [2/10], Loss: 0.0100
Gamma: 3

In [60]:
focal_mlp_predictions = {}

for gamma, trained_model in focal_mlp_models.items():

    trained_model.eval()

    predictions = []

    with torch.no_grad():

        for start in range(0, len(y_test_encoded), 128):

            X_batch = X_test_tfidf[start:start + 128].toarray().astype(np.float32)
            X_batch = torch.from_numpy(X_batch).to(device)

            outputs = trained_model(X_batch)

            batch_predictions = torch.argmax(outputs, dim=1)

            predictions.extend(batch_predictions.cpu().numpy())

    focal_mlp_predictions[gamma] = np.array(predictions)

    print(
        f"Gamma: {gamma}, "
        f"Prediction Count: {len(predictions)}"
    )

Gamma: 1.0, Prediction Count: 6215
Gamma: 2.0, Prediction Count: 6215
Gamma: 3.0, Prediction Count: 6215


In [62]:
focal_mlp_results = {}

for gamma, predictions in focal_mlp_predictions.items():

    metrics = calculate_metrics(
        y_test_encoded,
        predictions
    )

    focal_mlp_results[gamma] = metrics

    print(f"\nGamma: {gamma}")
    print("Accuracy:", round(metrics["accuracy"], 4))
    print("Balanced Accuracy:", round(metrics["balanced_accuracy"], 4))
    print("Macro F1:", round(metrics["macro_f1"], 4))
    print("Macro Specificity:", round(metrics["macro_specificity"], 4))

    print("\nClass-Based Results:")

    for i, class_name in enumerate(class_names):

        print(f"\n{class_name}")
        print("Precision:", round(metrics["class_precision"][i], 4))
        print("Recall:", round(metrics["class_recall"][i], 4))
        print("F1:", round(metrics["class_f1"][i], 4))
        print("Specificity:", round(metrics["class_specificity"][i], 4))

    print("\nConfusion Matrix:")
    print(metrics["confusion_matrix"])


Gamma: 1.0
Accuracy: 0.9971
Balanced Accuracy: 0.9529
Macro F1: 0.9628
Macro Specificity: 0.9991

Class-Based Results:

cmdi
Precision: 1.0
Recall: 0.7778
F1: 0.875
Specificity: 1.0

norm
Precision: 0.9984
Recall: 0.9972
F1: 0.9978
Specificity: 0.9975

path-traversal
Precision: 0.9667
Recall: 1.0
F1: 0.9831
Specificity: 0.9997

sqli
Precision: 0.9991
Recall: 0.9991
F1: 0.9991
Specificity: 0.9995

xss
Precision: 0.9298
Recall: 0.9907
F1: 0.9593
Specificity: 0.9987

Confusion Matrix:
[[  14    3    1    0    0]
 [   0 3850    1    2    8]
 [   0    0   58    0    0]
 [   0    2    0 2169    0]
 [   0    1    0    0  106]]

Gamma: 2.0
Accuracy: 0.9969
Balanced Accuracy: 0.9639
Macro F1: 0.9696
Macro Specificity: 0.999

Class-Based Results:

cmdi
Precision: 1.0
Recall: 0.8333
F1: 0.9091
Specificity: 1.0

norm
Precision: 0.9984
Recall: 0.9966
F1: 0.9975
Specificity: 0.9975

path-traversal
Precision: 0.9831
Recall: 1.0
F1: 0.9915
Specificity: 0.9998

sqli
Precision: 0.9991
Recall: 0.9991
F1

In [63]:
summary_rows = []

for gamma, metrics in focal_mlp_results.items():

    summary_rows.append({
        "Model": "MLP",
        "Loss Function": "Class-Weighted Focal Loss",
        "Gamma": gamma,
        "Accuracy": metrics["accuracy"],
        "Balanced Accuracy": metrics["balanced_accuracy"],
        "Macro F1": metrics["macro_f1"],
        "Macro Specificity": metrics["macro_specificity"]
    })

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    "../reports/mlp_focal_loss_results.csv",
    index=False,
    float_format="%.4f"
)

class_rows = []

for gamma, metrics in focal_mlp_results.items():

    for i, class_name in enumerate(class_names):

        class_rows.append({
            "Gamma": gamma,
            "Class": class_name,
            "Precision": metrics["class_precision"][i],
            "Recall": metrics["class_recall"][i],
            "F1 Score": metrics["class_f1"][i],
            "Specificity": metrics["class_specificity"][i]
        })

class_df = pd.DataFrame(class_rows)

class_df.to_csv(
    "../reports/mlp_focal_loss_class_metrics.csv",
    index=False,
    float_format="%.4f"
)

for gamma, metrics in focal_mlp_results.items():

    confusion_df = pd.DataFrame(
        metrics["confusion_matrix"],
        index=class_names,
        columns=class_names
    )

    confusion_df.index.name = "Actual / Predicted"

    confusion_df.to_csv(
        f"../reports/mlp_focal_loss_confusion_matrix_gamma_{gamma}.csv"
    )


display(summary_df)
display(class_df)

,Model,Loss Function,Gamma,Accuracy,Balanced Accuracy,Macro F1,Macro Specificity
0,MLP,Class-Weighted Focal Loss,1.0,0.997104,0.952932,0.962841,0.999064
1,MLP,Class-Weighted Focal Loss,2.0,0.996943,0.963940,0.969567,0.999031
2,MLP,Class-Weighted Focal Loss,3.0,0.967015,0.967182,0.810754,0.993273


,Gamma,Class,Precision,Recall,F1 Score,Specificity
0,1.0,cmdi,1.000000,0.777778,0.875000,1.000000
1,1.0,norm,0.998444,0.997151,0.997797,0.997451
2,1.0,path-traversal,0.966667,1.000000,0.983051,0.999675
3,1.0,sqli,0.999079,0.999079,0.999079,0.999505
4,1.0,xss,0.929825,0.990654,0.959276,0.998690
5,2.0,cmdi,1.000000,0.833333,0.909091,1.000000
6,2.0,norm,0.998443,0.996633,0.997537,0.997451
7,2.0,path-traversal,0.983051,1.000000,0.991453,0.999838
8,2.0,sqli,0.999079,0.999079,0.999079,0.999505
9,2.0,xss,0.913793,0.990654,0.950673,0.998363


##Random Oversampling

In [64]:
np.random.seed(42)

class_counts = np.bincount(y_train_encoded, minlength=num_classes)
target_count = class_counts.max()

oversampled_indices = []

for class_id in range(num_classes):
    class_indices = np.where(y_train_encoded == class_id)[0]

    if len(class_indices) < target_count:
        additional_indices = np.random.choice(
            class_indices,
            size=target_count - len(class_indices),
            replace=True
        )
        class_indices = np.concatenate([class_indices, additional_indices])

    oversampled_indices.extend(class_indices)

oversampled_indices = np.array(oversampled_indices)
np.random.shuffle(oversampled_indices)

X_train_ros = X_train_tfidf[oversampled_indices]
y_train_ros = y_train_encoded[oversampled_indices]

print("Original training shape:", X_train_tfidf.shape)
print("Oversampled training shape:", X_train_ros.shape)
print("Class distribution:", np.bincount(y_train_ros))
print("Test shape:", X_test_tfidf.shape)

Original training shape: (24852, 50000)
Oversampled training shape: (77215, 50000)
Class distribution: [15443 15443 15443 15443 15443]
Test shape: (6215, 50000)


In [65]:
ros_mlp_model = MLP(input_size, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(ros_mlp_model.parameters(), lr=0.001)

epochs = 10
batch_size = 128

for epoch in range(epochs):
    ros_mlp_model.train()
    epoch_loss = 0

    indices = np.random.permutation(len(y_train_ros))

    for start in range(0, len(indices), batch_size):
        batch_indices = indices[start:start + batch_size]

        X_batch = X_train_ros[batch_indices].toarray().astype(np.float32)
        y_batch = y_train_ros[batch_indices]

        X_batch = torch.tensor(X_batch, dtype=torch.float32, device=device)
        y_batch = torch.tensor(y_batch, dtype=torch.long, device=device)

        optimizer.zero_grad()

        outputs = ros_mlp_model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch + 1}/{epochs}, Loss: {epoch_loss:.4f}")

Epoch 1/10, Loss: 63.5565
Epoch 2/10, Loss: 3.9452
Epoch 3/10, Loss: 3.3329
Epoch 4/10, Loss: 3.1303
Epoch 5/10, Loss: 3.0063
Epoch 6/10, Loss: 2.8571
Epoch 7/10, Loss: 2.9088
Epoch 8/10, Loss: 2.7913
Epoch 9/10, Loss: 2.8271
Epoch 10/10, Loss: 2.7490


In [66]:
ros_mlp_model.eval()

ros_mlp_predictions = []

with torch.no_grad():
    for start in range(0, X_test_tfidf.shape[0], 128):
        X_batch = X_test_tfidf[start:start + 128].toarray().astype(np.float32)
        X_batch = torch.tensor(X_batch, dtype=torch.float32, device=device)

        outputs = ros_mlp_model(X_batch)
        predictions = torch.argmax(outputs, dim=1)

        ros_mlp_predictions.extend(predictions.cpu().numpy())

ros_mlp_predictions = np.array(ros_mlp_predictions)

print("Prediction count:", len(ros_mlp_predictions))

Prediction count: 6215


In [67]:
ros_mlp_results = calculate_metrics(
    y_test_encoded,
    ros_mlp_predictions
)

print("Accuracy:", ros_mlp_results["accuracy"])
print("Balanced Accuracy:", ros_mlp_results["balanced_accuracy"])
print("Macro F1:", ros_mlp_results["macro_f1"])
print("Macro Precision:", ros_mlp_results["macro_precision"])
print("Macro Recall:", ros_mlp_results["macro_recall"])
print("Macro Specificity:", ros_mlp_results["macro_specificity"])

for i, class_name in enumerate(class_names):
    print(
        class_name,
        "Precision:", ros_mlp_results["class_precision"][i],
        "Recall:", ros_mlp_results["class_recall"][i],
        "F1:", ros_mlp_results["class_f1"][i],
        "Specificity:", ros_mlp_results["class_specificity"][i]
    )

Accuracy: 0.9818181818181818
Balanced Accuracy: 0.9719876486343552
Macro F1: 0.8359510729504283
Macro Precision: 0.809291818903934
Macro Recall: 0.9719876486343552
Macro Specificity: 0.9961909781081012
cmdi Precision: 0.1415929203539823 Recall: 0.8888888888888888 F1: 0.2442748091603053 Specificity: 0.9843472648055511
norm Precision: 0.9994670929922729 Recall: 0.9715099715099715 F1: 0.985290254793801 Specificity: 0.9991503823279524
path-traversal Precision: 1.0 Recall: 1.0 F1: 1.0 Specificity: 1.0
sqli Precision: 0.9986194201564658 Recall: 0.9995393827729157 F1: 0.9990791896869246 Specificity: 0.9992581602373887
xss Precision: 0.9067796610169492 Recall: 1.0 F1: 0.9511111111111111 Specificity: 0.9981990831696136


In [68]:
ros_rows = []

ros_rows.append({
    "model": "MLP",
    "method": "Random Oversampling",
    "class": "overall",
    "accuracy": ros_mlp_results["accuracy"],
    "balanced_accuracy": ros_mlp_results["balanced_accuracy"],
    "macro_f1": ros_mlp_results["macro_f1"],
    "macro_precision": ros_mlp_results["macro_precision"],
    "macro_recall": ros_mlp_results["macro_recall"],
    "macro_specificity": ros_mlp_results["macro_specificity"]
})

for i, class_name in enumerate(class_names):
    ros_rows.append({
        "model": "MLP",
        "method": "Random Oversampling",
        "class": class_name,
        "precision": ros_mlp_results["class_precision"][i],
        "recall": ros_mlp_results["class_recall"][i],
        "f1": ros_mlp_results["class_f1"][i],
        "specificity": ros_mlp_results["class_specificity"][i]
    })

ros_df = pd.DataFrame(ros_rows)

ros_df.to_csv(
    "../reports/mlp_random_oversampling.csv",
    index=False
)



##SMOTE

In [69]:
def generate_smote_samples(X, y, target_count, k_neighbors=5):
    synthetic_features = []
    synthetic_labels = []

    for class_id in range(num_classes):
        class_indices = np.where(y == class_id)[0]
        class_count = len(class_indices)

        if class_count >= target_count:
            continue

        class_samples = X[class_indices]

        similarity = (class_samples @ class_samples.T).toarray()
        np.fill_diagonal(similarity, -np.inf)

        neighbor_indices = np.argsort(
            similarity,
            axis=1
        )[:, -min(k_neighbors, class_count - 1):]

        required_samples = target_count - class_count

        for _ in range(required_samples):
            sample_index = np.random.randint(class_count)
            neighbor_index = np.random.choice(
                neighbor_indices[sample_index]
            )

            sample = class_samples[sample_index]
            neighbor = class_samples[neighbor_index]

            interpolation = np.random.random()

            synthetic_sample = (
                sample + interpolation * (neighbor - sample)
            )

            synthetic_features.append(synthetic_sample)
            synthetic_labels.append(class_id)

    X_synthetic = vstack(synthetic_features, format="csr")
    y_synthetic = np.array(synthetic_labels, dtype=np.int64)

    X_resampled = vstack([X, X_synthetic], format="csr")
    y_resampled = np.concatenate([y, y_synthetic])

    return X_resampled, y_resampled


np.random.seed(42)

X_train_smote, y_train_smote = generate_smote_samples(
    X_train_tfidf,
    y_train_encoded,
    target_count=class_counts.max(),
    k_neighbors=5
)

print("Original shape:", X_train_tfidf.shape)
print("SMOTE shape:", X_train_smote.shape)
print("Class distribution:", np.bincount(y_train_smote))
print("Test shape:", X_test_tfidf.shape)

Original shape: (24852, 50000)
SMOTE shape: (77215, 50000)
Class distribution: [15443 15443 15443 15443 15443]
Test shape: (6215, 50000)


In [70]:
smote_mlp_model = MLP(input_size, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(smote_mlp_model.parameters(), lr=0.001)

epochs = 10
batch_size = 128

for epoch in range(epochs):
    smote_mlp_model.train()
    epoch_loss = 0

    indices = np.random.permutation(len(y_train_smote))

    for start in range(0, len(indices), batch_size):
        batch_indices = indices[start:start + batch_size]

        X_batch = X_train_smote[batch_indices].toarray().astype(np.float32)
        y_batch = y_train_smote[batch_indices]

        X_batch = torch.tensor(X_batch, dtype=torch.float32, device=device)
        y_batch = torch.tensor(y_batch, dtype=torch.long, device=device)

        optimizer.zero_grad()

        outputs = smote_mlp_model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch + 1}/{epochs}, Loss: {epoch_loss:.4f}")

Epoch 1/10, Loss: 57.2810
Epoch 2/10, Loss: 1.1985
Epoch 3/10, Loss: 0.8279
Epoch 4/10, Loss: 0.6517
Epoch 5/10, Loss: 0.5647
Epoch 6/10, Loss: 0.5201
Epoch 7/10, Loss: 0.4260
Epoch 8/10, Loss: 0.5364
Epoch 9/10, Loss: 0.3930
Epoch 10/10, Loss: 0.4405


In [71]:
smote_mlp_model.eval()

smote_mlp_predictions = []

with torch.no_grad():
    for start in range(0, X_test_tfidf.shape[0], 128):
        X_batch = X_test_tfidf[start:start + 128].toarray().astype(np.float32)
        X_batch = torch.tensor(X_batch, dtype=torch.float32, device=device)

        outputs = smote_mlp_model(X_batch)
        predictions = torch.argmax(outputs, dim=1)

        smote_mlp_predictions.extend(predictions.cpu().numpy())

smote_mlp_predictions = np.array(smote_mlp_predictions)

print("Prediction count:", len(smote_mlp_predictions))

Prediction count: 6215


In [72]:
smote_mlp_results = calculate_metrics(
    y_test_encoded,
    smote_mlp_predictions
)

print("Accuracy:", smote_mlp_results["accuracy"])
print("Balanced Accuracy:", smote_mlp_results["balanced_accuracy"])
print("Macro F1:", smote_mlp_results["macro_f1"])
print("Macro Precision:", smote_mlp_results["macro_precision"])
print("Macro Recall:", smote_mlp_results["macro_recall"])
print("Macro Specificity:", smote_mlp_results["macro_specificity"])

for i, class_name in enumerate(class_names):
    print(
        class_name,
        "Precision:", smote_mlp_results["class_precision"][i],
        "Recall:", smote_mlp_results["class_recall"][i],
        "F1:", smote_mlp_results["class_f1"][i],
        "Specificity:", smote_mlp_results["class_specificity"][i]
    )

Accuracy: 0.9964601769911504
Balanced Accuracy: 0.9545828312295377
Macro F1: 0.9400253338275621
Macro Precision: 0.926304694731097
Macro Recall: 0.9545828312295377
Macro Specificity: 0.9990234983310629
cmdi Precision: 0.7368421052631579 Recall: 0.7777777777777778 F1: 0.7567567567567567 Specificity: 0.9991931579796676
norm Precision: 0.998960498960499 Recall: 0.9955969955969955 F1: 0.9972759112725387 Specificity: 0.9983007646559049
path-traversal Precision: 0.9666666666666667 Recall: 1.0 F1: 0.983050847457627 Specificity: 0.9996751664771805
sqli Precision: 0.9986194201564658 Recall: 0.9995393827729157 F1: 0.9990791896869246 Specificity: 0.9992581602373887
xss Precision: 0.9304347826086956 Recall: 1.0 F1: 0.9639639639639639 Specificity: 0.9986902423051736


In [ ]:
smote_rows = []

smote_rows.append({
    "model": "MLP",
    "method": "SMOTE",
    "class": "overall",
    "accuracy": smote_mlp_results["accuracy"],
    "balanced_accuracy": smote_mlp_results["balanced_accuracy"],
    "macro_f1": smote_mlp_results["macro_f1"],
    "macro_precision": smote_mlp_results["macro_precision"],
    "macro_recall": smote_mlp_results["macro_recall"],
    "macro_specificity": smote_mlp_results["macro_specificity"]
})

for i, class_name in enumerate(class_names):
    smote_rows.append({
        "model": "MLP",
        "method": "SMOTE",
        "class": class_name,
        "precision": smote_mlp_results["class_precision"][i],
        "recall": smote_mlp_results["class_recall"][i],
        "f1": smote_mlp_results["class_f1"][i],
        "specificity": smote_mlp_results["class_specificity"][i]
    })

smote_df = pd.DataFrame(smote_rows)

smote_df.to_csv(
    "../reports/mlp_smote.csv",
    index=False
)



MLP SMOTE results saved successfully.


##Borderline-SMOTE

In [74]:
def find_danger_samples(X, y, m_neighbors=10):
    danger_samples = {}

    row_norms = np.sqrt(X.multiply(X).sum(axis=1)).A1
    class_counts = np.bincount(y, minlength=num_classes)
    minority_classes = np.where(class_counts < class_counts.max())[0]

    for class_id in minority_classes:
        class_indices = np.where(y == class_id)[0]
        danger_indices = []

        for sample_index in class_indices:
            similarities = (X[sample_index] @ X.T).toarray().ravel()
            denominators = row_norms[sample_index] * row_norms

            similarities = np.divide(
                similarities,
                denominators,
                out=np.zeros_like(similarities),
                where=denominators != 0
            )

            similarities[sample_index] = -np.inf

            neighbor_count = min(m_neighbors, len(y) - 1)
            neighbor_indices = np.argpartition(
                similarities,
                -neighbor_count
            )[-neighbor_count:]

            other_class_count = np.sum(y[neighbor_indices] != class_id)

            if neighbor_count / 2 <= other_class_count < neighbor_count:
                danger_indices.append(sample_index)

        danger_samples[class_id] = np.array(danger_indices)

    return danger_samples


danger_samples = find_danger_samples(
    X_train_tfidf,
    y_train_encoded,
    m_neighbors=10
)

for class_id, indices in danger_samples.items():
    print(f"{class_names[class_id]}: {len(indices)} danger samples")

cmdi: 19 danger samples
path-traversal: 10 danger samples
sqli: 335 danger samples
xss: 6 danger samples


In [75]:
def generate_borderline_smote(X, y, danger_samples, target_count, k_neighbors=5):
    synthetic_features = []
    synthetic_labels = []

    for class_id, danger_indices in danger_samples.items():
        class_indices = np.where(y == class_id)[0]
        class_count = len(class_indices)

        if class_count >= target_count:
            continue

        class_samples = X[class_indices]

        similarity = (class_samples @ class_samples.T).toarray()

        row_norms = np.sqrt(class_samples.multiply(class_samples).sum(axis=1)).A1
        denominators = row_norms[:, None] * row_norms[None, :]

        similarity = np.divide(
            similarity,
            denominators,
            out=np.zeros_like(similarity),
            where=denominators != 0
        )

        np.fill_diagonal(similarity, -np.inf)

        neighbor_count = min(k_neighbors, class_count - 1)
        neighbor_indices = np.argsort(
            similarity,
            axis=1
        )[:, -neighbor_count:]

        if len(danger_indices) == 0:
            danger_indices = class_indices

        danger_local_indices = np.searchsorted(class_indices, danger_indices)
        required_samples = target_count - class_count

        for _ in range(required_samples):
            sample_local_index = np.random.choice(danger_local_indices)
            neighbor_local_index = np.random.choice(
                neighbor_indices[sample_local_index]
            )

            sample = class_samples[sample_local_index]
            neighbor = class_samples[neighbor_local_index]

            interpolation = np.random.random()

            synthetic_sample = (
                sample + interpolation * (neighbor - sample)
            )

            synthetic_features.append(synthetic_sample)
            synthetic_labels.append(class_id)

    X_synthetic = vstack(synthetic_features, format="csr")
    y_synthetic = np.array(synthetic_labels, dtype=np.int64)

    X_resampled = vstack([X, X_synthetic], format="csr")
    y_resampled = np.concatenate([y, y_synthetic])

    return X_resampled, y_resampled


np.random.seed(42)

X_train_borderline, y_train_borderline = generate_borderline_smote(
    X_train_tfidf,
    y_train_encoded,
    danger_samples,
    target_count=class_counts.max(),
    k_neighbors=5
)

print("Original shape:", X_train_tfidf.shape)
print("Borderline-SMOTE shape:", X_train_borderline.shape)
print("Class distribution:", np.bincount(y_train_borderline))

Original shape: (24852, 50000)
Borderline-SMOTE shape: (77215, 50000)
Class distribution: [15443 15443 15443 15443 15443]


In [76]:
borderline_mlp_model = MLP(input_size, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(borderline_mlp_model.parameters(), lr=0.001)

epochs = 10
batch_size = 128

for epoch in range(epochs):
    borderline_mlp_model.train()
    epoch_loss = 0

    indices = np.random.permutation(len(y_train_borderline))

    for start in range(0, len(indices), batch_size):
        batch_indices = indices[start:start + batch_size]

        X_batch = X_train_borderline[batch_indices].toarray().astype(np.float32)
        y_batch = y_train_borderline[batch_indices]

        X_batch = torch.tensor(X_batch, dtype=torch.float32, device=device)
        y_batch = torch.tensor(y_batch, dtype=torch.long, device=device)

        optimizer.zero_grad()

        outputs = borderline_mlp_model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch + 1}/{epochs}, Loss: {epoch_loss:.4f}")

Epoch 1/10, Loss: 58.8204
Epoch 2/10, Loss: 1.0773
Epoch 3/10, Loss: 0.7718
Epoch 4/10, Loss: 0.5212
Epoch 5/10, Loss: 0.4676
Epoch 6/10, Loss: 0.3650
Epoch 7/10, Loss: 0.3288
Epoch 8/10, Loss: 0.3623
Epoch 9/10, Loss: 0.3365
Epoch 10/10, Loss: 0.2916


In [77]:
borderline_mlp_model.eval()

borderline_mlp_predictions = []

with torch.no_grad():
    for start in range(0, X_test_tfidf.shape[0], 128):
        X_batch = X_test_tfidf[start:start + 128].toarray().astype(np.float32)
        X_batch = torch.tensor(X_batch, dtype=torch.float32, device=device)

        outputs = borderline_mlp_model(X_batch)
        predictions = torch.argmax(outputs, dim=1)

        borderline_mlp_predictions.extend(predictions.cpu().numpy())

borderline_mlp_predictions = np.array(borderline_mlp_predictions)

print("Prediction count:", len(borderline_mlp_predictions))

Prediction count: 6215


In [78]:
borderline_mlp_results = calculate_metrics(
    y_test_encoded,
    borderline_mlp_predictions
)

print("Accuracy:", borderline_mlp_results["accuracy"])
print("Balanced Accuracy:", borderline_mlp_results["balanced_accuracy"])
print("Macro F1:", borderline_mlp_results["macro_f1"])
print("Macro Precision:", borderline_mlp_results["macro_precision"])
print("Macro Recall:", borderline_mlp_results["macro_recall"])
print("Macro Specificity:", borderline_mlp_results["macro_specificity"])

for i, class_name in enumerate(class_names):
    print(
        class_name,
        "Precision:", borderline_mlp_results["class_precision"][i],
        "Recall:", borderline_mlp_results["class_recall"][i],
        "F1:", borderline_mlp_results["class_f1"][i],
        "Specificity:", borderline_mlp_results["class_specificity"][i]
    )

Accuracy: 0.9971037811745777
Balanced Accuracy: 0.9529323492164167
Macro F1: 0.9592436038210309
Macro Precision: 0.9686982678722895
Macro Recall: 0.9529323492164167
Macro Specificity: 0.9990481574046479
cmdi Precision: 0.9333333333333333 Recall: 0.7777777777777778 F1: 0.8484848484848485 Specificity: 0.9998386315959336
norm Precision: 0.9984439834024896 Recall: 0.9971509971509972 F1: 0.9977970714008034 Specificity: 0.9974511469838573
path-traversal Precision: 0.9666666666666667 Recall: 1.0 F1: 0.983050847457627 Specificity: 0.9996751664771805
sqli Precision: 0.9986187845303868 Recall: 0.9990787655458314 F1: 0.9988487220815104 Specificity: 0.9992581602373887
xss Precision: 0.9464285714285714 Recall: 0.9906542056074766 F1: 0.9680365296803651 Specificity: 0.9990176817288802


In [80]:
borderline_rows = []

borderline_rows.append({
    "model": "MLP",
    "method": "Borderline-SMOTE",
    "class": "overall",
    "accuracy": borderline_mlp_results["accuracy"],
    "balanced_accuracy": borderline_mlp_results["balanced_accuracy"],
    "macro_f1": borderline_mlp_results["macro_f1"],
    "macro_precision": borderline_mlp_results["macro_precision"],
    "macro_recall": borderline_mlp_results["macro_recall"],
    "macro_specificity": borderline_mlp_results["macro_specificity"]
})

for i, class_name in enumerate(class_names):
    borderline_rows.append({
        "model": "MLP",
        "method": "Borderline-SMOTE",
        "class": class_name,
        "precision": borderline_mlp_results["class_precision"][i],
        "recall": borderline_mlp_results["class_recall"][i],
        "f1": borderline_mlp_results["class_f1"][i],
        "specificity": borderline_mlp_results["class_specificity"][i]
    })

borderline_df = pd.DataFrame(borderline_rows)

borderline_df.to_csv(
    "../reports/mlp_borderline_smote.csv",
    index=False
)

